# Tessera-PQC Tutorial 3: Countermeasures

This notebook covers countermeasures against side-channel attacks:
- **Masking**: Splitting sensitive values into random shares
- **Shuffling**: Randomizing operation order
- **Hiding**: Adding noise and using dual-rail logic

In [ ]:
import tessera as t
import numpy as np
import matplotlib.pyplot as plt

from tessera.countermeasures import (
    BooleanMasking, ArithmeticMasking,
    SecurePermutation, ShuffledNTT,
    NoiseInjector, DualRailLogic, TraceHider
)

np.random.seed(42)

## 1. Boolean Masking

Boolean masking splits a sensitive value `x` into random shares such that:
`x = s0 XOR s1 XOR ... XOR sn`

Each share individually reveals nothing about `x`.

In [ ]:
# Create Boolean masking instance
bool_masking = BooleanMasking(bit_width=64)

# Original sensitive data
sensitive_data = np.array([0x12345678, 0xDEADBEEF, 0xCAFEBABE], dtype=np.uint64)
print(f"Original data: {[hex(x) for x in sensitive_data]}")

# Create 2 shares (first-order masking)
masked = bool_masking.mask(sensitive_data, num_shares=2)

print(f"\nShare 0: {[hex(x) for x in masked.shares[0]]}")
print(f"Share 1: {[hex(x) for x in masked.shares[1]]}")

# Verify reconstruction
reconstructed = masked.unmask()
print(f"\nReconstructed: {[hex(x) for x in reconstructed]}")
print(f"Correct: {np.array_equal(sensitive_data, reconstructed)}")

In [ ]:
# Higher-order masking (more shares = more security)
for num_shares in [2, 3, 4, 5]:
    masked = bool_masking.mask(sensitive_data, num_shares=num_shares)
    reconstructed = masked.unmask()
    print(f"{num_shares} shares (order {num_shares-1}): correct={np.array_equal(sensitive_data, reconstructed)}")

### Masking Effectiveness

Let's demonstrate how masking affects the leakage.

In [ ]:
from tessera.leakage import HammingWeightModel

hw_model = HammingWeightModel(64)

# Simulate leakage with and without masking
n_samples = 1000
value = np.array([0x55555555], dtype=np.uint64)  # Fixed value with HW=16

# Without masking - leakage is constant
unmasked_leakage = [hw_model(int(value[0])) for _ in range(n_samples)]

# With masking - leakage varies randomly
masked_leakage_share0 = []
masked_leakage_share1 = []
for _ in range(n_samples):
    masked = bool_masking.mask(value, num_shares=2)
    masked_leakage_share0.append(hw_model(int(masked.shares[0][0])))
    masked_leakage_share1.append(hw_model(int(masked.shares[1][0])))

In [ ]:
# Plot leakage distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(unmasked_leakage, bins=20, alpha=0.7, color='red')
axes[0].set_xlabel("Hamming Weight")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Unmasked Leakage\n(constant HW={unmasked_leakage[0]})")

axes[1].hist(masked_leakage_share0, bins=20, alpha=0.7, color='blue')
axes[1].set_xlabel("Hamming Weight")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Masked Share 0 Leakage\n(random, mean={np.mean(masked_leakage_share0):.1f})")

axes[2].hist(masked_leakage_share1, bins=20, alpha=0.7, color='green')
axes[2].set_xlabel("Hamming Weight")
axes[2].set_ylabel("Count")
axes[2].set_title(f"Masked Share 1 Leakage\n(random, mean={np.mean(masked_leakage_share1):.1f})")

plt.tight_layout()
plt.show()

print(f"Unmasked leakage variance: {np.var(unmasked_leakage):.4f}")
print(f"Share 0 leakage variance:  {np.var(masked_leakage_share0):.4f}")
print(f"Share 1 leakage variance:  {np.var(masked_leakage_share1):.4f}")

## 2. Arithmetic Masking

Arithmetic masking uses modular addition instead of XOR:
`x = (s0 + s1 + ... + sn) mod q`

This is useful for lattice-based crypto where operations are modular arithmetic.

In [ ]:
# Kyber uses modulus q=3329
arith_masking = ArithmeticMasking(modulus=3329)

# Polynomial coefficients (mod 3329)
coefficients = np.array([1234, 2567, 890, 3000], dtype=np.int64)
print(f"Original coefficients: {coefficients.tolist()}")

# Create shares
masked = arith_masking.mask(coefficients, num_shares=2)

print(f"\nShare 0: {masked.shares[0].tolist()}")
print(f"Share 1: {masked.shares[1].tolist()}")

# Verify: (share0 + share1) mod 3329 == original
reconstructed = masked.unmask()
print(f"\nReconstructed: {reconstructed.tolist()}")
print(f"Correct: {np.array_equal(coefficients, reconstructed)}")

## 3. Shuffling

Shuffling randomizes the order of operations to prevent attacks
that rely on temporal patterns.

In [ ]:
# Create secure permutation generator
perm = SecurePermutation(size=8)

# Generate random permutations
print("Random permutations:")
for i in range(5):
    indices = perm.generate()
    print(f"  {indices.tolist()}")

In [ ]:
# Demonstrate shuffling effect on operations
operations = ['op_A', 'op_B', 'op_C', 'op_D', 'op_E', 'op_F', 'op_G', 'op_H']

print("Original order:", operations)
print("\nShuffled orders:")

for i in range(5):
    shuffled = t.apply_shuffling(operations)
    print(f"  Run {i+1}: {shuffled}")

### Shuffled NTT

NTT butterfly operations can be shuffled to randomize leakage patterns.

In [ ]:
# Create shuffled NTT instance
shuffled_ntt = ShuffledNTT(n=256, q=3329)

# Random polynomial
poly = np.random.randint(0, 3329, 256, dtype=np.int64)

# Apply shuffled NTT
poly_ntt = shuffled_ntt.ntt(poly.copy())
poly_recovered = shuffled_ntt.intt(poly_ntt.copy())

print(f"Original first 8 coeffs:    {poly[:8].tolist()}")
print(f"Recovered first 8 coeffs:   {poly_recovered[:8].tolist()}")
print(f"Round-trip correct: {np.array_equal(poly, poly_recovered)}")

## 4. Hiding Countermeasures

Hiding adds noise or uses special logic to obscure the power consumption.

In [ ]:
# Noise Injection
noise_injector = NoiseInjector(noise_amplitude=2.0)

# Original trace (clean signal)
clean_trace = np.sin(np.linspace(0, 4*np.pi, 100)) * 5 + 10

# Apply noise injection multiple times
noisy_traces = [noise_injector.inject(clean_trace.copy()) for _ in range(5)]

plt.figure(figsize=(12, 4))
plt.plot(clean_trace, 'b-', linewidth=2, label='Original')
for i, noisy in enumerate(noisy_traces):
    plt.plot(noisy, alpha=0.5, label=f'Noisy {i+1}')
plt.xlabel("Sample")
plt.ylabel("Power")
plt.title("Noise Injection Countermeasure")
plt.legend()
plt.show()

In [ ]:
# Dual-Rail Logic
# Encodes each bit as two complementary wires to balance power consumption
dual_rail = DualRailLogic(bit_width=8)

# Encode values
values = np.array([0x00, 0xFF, 0x55, 0xAA, 0x12], dtype=np.uint8)

print("Dual-Rail Encoding:")
print("-" * 50)
for val in values:
    true_rail, complement_rail = dual_rail.encode(int(val))
    hw_true = bin(true_rail).count('1')
    hw_comp = bin(complement_rail).count('1')
    print(f"Value: 0x{val:02X} -> True: 0x{true_rail:02X} (HW={hw_true}), "
          f"Complement: 0x{complement_rail:02X} (HW={hw_comp}), Total HW: {hw_true + hw_comp}")

In [ ]:
# Trace Hider - combines multiple hiding techniques
trace_hider = TraceHider(
    add_noise=True,
    noise_amplitude=1.5,
    add_jitter=True,
    max_jitter=3
)

# Generate multiple hidden versions of the same trace
original_trace = np.array([1, 2, 5, 8, 10, 8, 5, 2, 1], dtype=np.float64)

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.plot(original_trace, 'b-o', linewidth=2, markersize=8, label='Original')
plt.xlabel("Sample")
plt.ylabel("Power")
plt.title("Original Trace")
plt.legend()

plt.subplot(122)
for i in range(5):
    hidden = trace_hider.hide(original_trace.copy())
    plt.plot(hidden, '-o', alpha=0.7, label=f'Hidden {i+1}')
plt.xlabel("Sample")
plt.ylabel("Power")
plt.title("Hidden Traces (noise + jitter)")
plt.legend()

plt.tight_layout()
plt.show()

## 5. Countermeasure Effectiveness

Let's compare attack success with and without countermeasures.

In [ ]:
# Generate traces WITHOUT countermeasures
data_unprotected = t.generate_traces(
    n_traces=500,
    trace_length=32,
    key_bytes=4,
    noise_level=0.5,
    seed=42
)

# Attack unprotected traces
result_unprotected = t.run_cpa(
    data_unprotected['traces'],
    data_unprotected['plaintexts'],
    key_bytes=[0, 1, 2, 3]
)

correct_unprotected = np.sum(
    result_unprotected['recovered_key'] == data_unprotected['key']
)

print(f"Unprotected: {correct_unprotected}/4 bytes recovered")
print(f"True key:      {data_unprotected['key'].tolist()}")
print(f"Recovered key: {result_unprotected['recovered_key'].tolist()}")

In [ ]:
# Generate traces WITH hiding countermeasure (add noise)
traces_protected = data_unprotected['traces'].copy()
noise_injector = NoiseInjector(noise_amplitude=3.0)

for i in range(len(traces_protected)):
    traces_protected[i] = noise_injector.inject(traces_protected[i])

# Attack protected traces
result_protected = t.run_cpa(
    traces_protected,
    data_unprotected['plaintexts'],
    key_bytes=[0, 1, 2, 3]
)

correct_protected = np.sum(
    result_protected['recovered_key'] == data_unprotected['key']
)

print(f"Protected (noise injection): {correct_protected}/4 bytes recovered")
print(f"Recovered key: {result_protected['recovered_key'].tolist()}")

In [ ]:
# Compare SNR
labels = data_unprotected['plaintexts'][:, 0] ^ data_unprotected['key'][0]

snr_unprotected = t.compute_snr(data_unprotected['traces'], labels)
snr_protected = t.compute_snr(traces_protected, labels)

plt.figure(figsize=(12, 4))

plt.subplot(121)
plt.plot(snr_unprotected, 'r-', linewidth=1.5, label='Unprotected')
plt.plot(snr_protected, 'b-', linewidth=1.5, label='Protected')
plt.xlabel("Sample")
plt.ylabel("SNR")
plt.title("SNR Comparison")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(122)
categories = ['Unprotected', 'Protected']
max_snrs = [np.max(snr_unprotected), np.max(snr_protected)]
plt.bar(categories, max_snrs, color=['red', 'blue'], alpha=0.7)
plt.ylabel("Max SNR")
plt.title("Maximum SNR")

plt.tight_layout()
plt.show()

print(f"Max SNR (unprotected): {np.max(snr_unprotected):.4f}")
print(f"Max SNR (protected):   {np.max(snr_protected):.4f}")
print(f"SNR reduction:         {(1 - np.max(snr_protected)/np.max(snr_unprotected))*100:.1f}%")

## 6. Using the High-Level API

Tessera provides convenient functions for applying countermeasures.

In [ ]:
# Apply masking using high-level API
sensitive_values = np.array([100, 200, 300, 400], dtype=np.uint64)

# Boolean masking
masked_bool = t.apply_masking(
    sensitive_values,
    num_shares=3,
    mask_type="boolean"
)

print("Boolean Masking (3 shares):")
print(f"  Original:  {masked_bool['original'].tolist()}")
print(f"  Share 0:   {masked_bool['shares'][0].tolist()}")
print(f"  Share 1:   {masked_bool['shares'][1].tolist()}")
print(f"  Share 2:   {masked_bool['shares'][2].tolist()}")

In [ ]:
# Apply shuffling
operations = ['load', 'ntt_stage1', 'ntt_stage2', 'multiply', 'intt', 'store']

print("Operation Shuffling:")
print(f"  Original:  {operations}")
for i in range(3):
    shuffled = t.apply_shuffling(operations)
    print(f"  Shuffled {i+1}: {shuffled}")

## Summary

In this notebook, you learned about three main categories of countermeasures:

### 1. Masking
- **Boolean masking**: Splits values using XOR
- **Arithmetic masking**: Splits values using modular addition
- Higher-order masking (more shares) provides more security

### 2. Shuffling
- Randomizes operation order
- Prevents temporal pattern exploitation
- Can be applied to NTT butterfly operations

### 3. Hiding
- **Noise injection**: Adds random noise to traces
- **Dual-rail logic**: Balances power consumption
- **Jitter**: Randomizes timing

**Key Takeaway**: Combining multiple countermeasures provides defense in depth.

### Trade-offs

| Countermeasure | Security | Performance Cost | Implementation Complexity |
|----------------|----------|------------------|---------------------------|
| Boolean Masking (order 1) | Moderate | ~2x | Low |
| Boolean Masking (order 2+) | High | ~n*x | Moderate |
| Arithmetic Masking | Moderate | ~2-3x | Moderate |
| Shuffling | Low-Moderate | ~1.1-1.5x | Low |
| Noise Injection | Low-Moderate | ~1x | Very Low |
| Dual-Rail Logic | Moderate | ~2x | High |